In [0]:
%pip install databricks-sdk
%pip install databricks-labs-lsql

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("warehouse_id_env", "", "warehouse_id_env")

warehouse_id_env = dbutils.widgets.get("warehouse_id_env")

In [0]:
import json, os, pathlib, sys
from databricks.sdk import WorkspaceClient
from databricks.labs.lsql.core import StatementExecutionExt

def main():
    current_dir = pathlib.Path(os.getcwd())
    config_path = current_dir / "lsql.json"
    cfg = json.loads(config_path.read_text())

    wh_id = warehouse_id_env
    if not wh_id:
        print(f"Missing env var {cfg['warehouse_id_env']}", file=sys.stderr)
        sys.exit(2)

    w = WorkspaceClient()
    see = StatementExecutionExt(w, warehouse_id=wh_id)

    see.execute(f"USE CATALOG {cfg['catalog']}")
    see.execute(f"CREATE SCHEMA IF NOT EXISTS {cfg['catalog']}.{cfg['schema']}")

    for step in cfg["steps"]:
        path = pathlib.Path(os.getcwd()) / step["path"]
        sql = path.read_text()
        print(f"--> Executing: {step['name']} ({path})")
        see.execute(sql)

if __name__ == "__main__":
    main()